In [ ]:
# 1. Uninstall broken libraries
!pip uninstall -y torchcodec

# 2. Install the "Golden" version of Transformers (Fixes the 'visual' error)
!pip install "transformers==4.48.3" "huggingface_hub>=0.24.0"

# 3. Install LLaMA-Factory and helpers
!pip install llamafactory[metrics]
!pip install qwen-vl-utils decord bitsandbytes peft scipy

print("✅ Environment Ready!")

Found existing installation: torchcodec 0.8.0
Uninstalling torchcodec-0.8.0:
  Successfully uninstalled torchcodec-0.8.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 65.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 98.4 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingface_hub-1.4.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import json
import os
from google.colab import drive

# 1. Mount Drive
drive.mount('/content/drive')

# 2. Process the Dataset
input_file = "football_dataset.json"  # <--- Make sure you uploaded this!
output_file = "football_drive_clean.json"

if os.path.exists(input_file):
    with open(input_file, 'r') as f:
        data = json.load(f)

    drive_data = []
    print(f"Processing {len(data)} entries...")

    for entry in data:
        # Extract info
        msgs = entry['messages']
        raw_path = msgs[0]['content'][0]['video']
        assistant_content = msgs[1]['content']

        # Fix Path Logic (Point to Drive)
        if "/content/drive/" not in raw_path:
             filename = raw_path.split("/")[-1]
             # ADJUST THIS PATH if your videos are in a different Drive folder
             clean_path = f"/content/drive/MyDrive/labeled/{filename}"
        else:
             # Fix "file.mp4/file.mp4" double path error
             head, tail = os.path.split(raw_path)
             if head.endswith(tail):
                 clean_path = head
             else:
                 clean_path = raw_path

        # Create Entry
        new_entry = {
            "conversations": [
                {
                    "from": "human",
                    "value": "<video>\nIdentify the football action in this video and rate its quality from 1 to 10."
                },
                {
                    "from": "gpt",
                    "value": assistant_content
                }
            ],
            "videos": [clean_path]
        }
        drive_data.append(new_entry)

    # Save the Clean File
    with open(output_file, 'w') as f:
        json.dump(drive_data, f, indent=2)
    print(f"✅ Saved clean dataset: {output_file}")

    # 3. CREATE THE MISSING REGISTRY FILE MANUALLY
    # This fixes the "FileNotFoundError: dataset_info.json"
    os.makedirs("data", exist_ok=True)

    dataset_entry = {
      "football_drive_clean": {
        "file_name": f"/content/{output_file}",
        "formatting": "sharegpt",
        "columns": {
          "messages": "conversations",
          "videos": "videos"
        }
      }
    }

    with open("data/dataset_info.json", "w", encoding="utf-8") as f:
        json.dump(dataset_entry, f, indent=2)

    print("✅ Successfully registered dataset in 'data/dataset_info.json'")

else:
    print("❌ ERROR: Please upload 'football_dataset.json' to the Colab files first!")

Mounted at /content/drive
Processing 2851 entries...
✅ Saved clean dataset: football_drive_clean.json
✅ Successfully registered dataset in 'data/dataset_info.json'


In [ ]:
# Run Training
!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path Qwen/Qwen2-VL-2B-Instruct \
    --dataset football_drive_clean \
    --dataset_dir data \
    --template qwen2_vl \
    --finetuning_type lora \
    --lora_target all \
    --output_dir "/content/drive/MyDrive/football_checkpoints_REAL" \
    --overwrite_output_dir \
    --warmup_ratio 0.1 \
    --weight_decay 0.1 \
    --learning_rate 2e-4 \
    --num_train_epochs 3.0 \
    --max_grad_norm 1.0 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --quantization_bit 4 \
    --gradient_checkpointing True \
    --logging_steps 10 \
    --save_steps 50 \
    --save_total_limit 2 \
    --plot_loss \
    --bf16 False \
    --fp16 True \
    --preprocessing_num_workers 1 \
    --dataloader_num_workers 0

2026-02-17 15:56:56.187242: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-17 15:56:56.192640: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-17 15:56:56.205483: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771343816.228686    1464 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771343816.235925    1464 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771343816.254081    1464 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [ ]:
import json
import os

# 1. Load the dirty dataset
input_file = "football_drive_clean.json"
output_file = "football_drive_final.json" # New final name

with open(input_file, 'r') as f:
    data = json.load(f)

print(f"Checking {len(data)} videos... (This takes about 30 seconds)")

clean_data = []
missing_count = 0

for entry in data:
    video_path = entry['videos'][0]

    # 2. Check if the file ACTUALLY exists on the disk
    if os.path.exists(video_path):
        clean_data.append(entry)
    else:
        # If missing, skip it
        missing_count += 1
        # Optional: Print first 3 missing to confirm it's working
        if missing_count <= 3:
             print(f"❌ Removing missing file: {video_path}")

# 3. Save the TRULY clean file
with open(output_file, 'w') as f:
    json.dump(clean_data, f, indent=2)

print("-" * 30)
print(f"Done! Removed {missing_count} ghost files.")
print(f"Final Dataset Size: {len(clean_data)}")

# 4. UPDATE THE REGISTRY to point to this new file
dataset_entry = {
  "football_drive_final": {
    "file_name": f"/content/{output_file}",
    "formatting": "sharegpt",
    "columns": {
      "messages": "conversations",
      "videos": "videos"
    }
  }
}

# Overwrite the registry file
os.makedirs("data", exist_ok=True)
with open("data/dataset_info.json", "w", encoding="utf-8") as f:
    json.dump(dataset_entry, f, indent=2)

print("✅ Registry updated to use 'football_drive_final'")

Checking 2851 videos... (This takes about 30 seconds)
❌ Removing missing file: /content/drive/MyDrive/labeled/pass/pass_video_1.avi
❌ Removing missing file: /content/drive/MyDrive/labeled/pass/pass_video_1.avi
❌ Removing missing file: /content/drive/MyDrive/labeled/pass/pass_video_1.avi
------------------------------
Done! Removed 80 ghost files.
Final Dataset Size: 2771
✅ Registry updated to use 'football_drive_final'


In [ ]:
!llamafactory-cli train \
    --stage sft \
    --do_train \
    --model_name_or_path Qwen/Qwen2-VL-2B-Instruct \
    --dataset football_drive_final \
    --dataset_dir data \
    --template qwen2_vl \
    --finetuning_type lora \
    --lora_target all \
    --output_dir "/content/drive/MyDrive/football_checkpoints_REAL" \
    --overwrite_output_dir \
    --warmup_ratio 0.1 \
    --weight_decay 0.1 \
    --learning_rate 2e-4 \
    --num_train_epochs 3.0 \
    --max_grad_norm 1.0 \
    --per_device_train_batch_size 1 \
    --gradient_accumulation_steps 8 \
    --quantization_bit 4 \
    --gradient_checkpointing True \
    --logging_steps 10 \
    --save_steps 50 \
    --save_total_limit 2 \
    --plot_loss \
    --bf16 False \
    --fp16 True \
    --preprocessing_num_workers 1 \
    --dataloader_num_workers 0

2026-02-17 16:14:47.982803: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-17 16:14:47.993712: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-17 16:14:48.027543: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1771344888.080325    7170 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1771344888.096509    7170 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1771344888.142665    7170 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin